# 10 · From frozen models to competition predictions

**Question:** Can the selected recommender produce every required prediction, with a traceable model and data history?

The default mode runs the real native models on eight compact, verified competition sessions. Its output must match the corresponding rows from the complete run exactly. The full mode invokes the same resumable inference pipeline on every competition session. No hidden test labels or leaderboard score are available.


In [ ]:
import hashlib
import json
import os
import subprocess
import time
from datetime import UTC, datetime
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display

STARTED = time.perf_counter()
FULL = os.environ.get("OTTO_FULL_INFERENCE") == "1"
location = Path.cwd().resolve()
ROOT = next(p for p in (location, *location.parents)
            if (p / "pyproject.toml").is_file()
            or ((p / "reports").is_dir() and (p / "configs").is_dir()))
print(datetime.now(UTC).isoformat(), "mode=", "full competition inference" if FULL else "verified native-model replay")


## Model and data contract

Model weights are fixed by chronological model selection and evaluated before competition inference. Retrieval and historical statistics can then be refreshed from the official training events that precede the competition inputs. This refresh is a deployment operation; its outputs do not enter the reported temporal evaluation.

Set `OTTO_FULL_INFERENCE=1` in the managed execution environment to generate the complete prediction file. The managed run executes this mode with verified inputs, the locked project interpreter, and durable part checkpoints. Default replay requires no AWS access or training data.


In [ ]:
if FULL:
    command = [str(ROOT / ".venv/bin/python"), str(ROOT / "scripts/run_inference.py"),
               "--stage", "predict", "--models", str(ROOT / "artifacts/research"),
               "--test", str(ROOT / "artifacts/test"), "--output", str(ROOT / "artifacts/inference"),
               "--threads", os.environ.get("OTTO_PREDICTION_THREADS", "4"),
               "--workers", os.environ.get("OTTO_PREDICTION_WORKERS", "1")]
    checkpoint_uri = os.environ.get("OTTO_PREDICTION_CHECKPOINT_URI")
    if checkpoint_uri:
        command += ["--checkpoint-uri", checkpoint_uri, "--owner-account", os.environ["OTTO_OWNER_ACCOUNT"],
                    "--region", os.environ["OTTO_AWS_REGION"]]
    completed = subprocess.run(command, cwd=ROOT, check=True, capture_output=True, text=True)
    print(completed.stdout)
    full = json.loads((ROOT / "artifacts/inference/prediction/manifest.json").read_text())
    destination = ROOT / "artifacts/inference/prediction/submission.csv.gz"
    preview = pd.read_csv(destination, nrows=9)
    assert full["status"] == "passed"
else:
    bundle = ROOT / "reports/research/inference_replay"
    manifest = json.loads((bundle / "manifest.json").read_text())
    assert manifest["status"] == "passed"
    for name, expected in manifest["files"].items():
        assert hashlib.sha256((bundle / name).read_bytes()).hexdigest() == expected, name
    candidates = pd.read_parquet(bundle / "features.parquet")
    records = []
    objectives = ("clicks", "carts", "orders")
    models = {o: lgb.Booster(model_file=str(bundle / manifest["models"][o]["path"])) for o in objectives}
    for session, query in candidates.groupby("session", sort=True):
        aids = query["aid"].to_numpy()
        for objective in objectives:
            names = manifest["models"][objective]["features"]
            assert models[objective].feature_name() == names
            score = models[objective].predict(query[names].to_numpy(dtype=np.float32), num_threads=1)
            order = np.lexsort((aids, -score))[:20]
            records.append((f"{session}_{objective}", " ".join(map(str, aids[order]))))
    replay = pd.DataFrame(records, columns=["session_type", "labels"])
    destination = ROOT / "artifacts/inference_replay.csv"
    destination.parent.mkdir(parents=True, exist_ok=True)
    replay.to_csv(destination, index=False)
    assert destination.read_bytes() == (bundle / "expected.csv").read_bytes()
    full = manifest["full_prediction"]
    preview = replay.head(9)
    print(f"Exact replay passed: {len(replay):,} rows, {manifest['sessions']} real competition sessions")


## Coverage and output validation

Every observed session must appear exactly once for each of clicks, carts, and orders. Each recommendation list contains 20 unique nonnegative item IDs. The full validator checks the exact session ledger, objective coverage, duplicate rows, list lengths, and the final file checksum. Partial or incompatible artifacts cannot certify completion.


In [ ]:
display(pd.DataFrame([
    ("Complete competition sessions", f"{full['sessions']:,}"),
    ("Required task rows", f"{full['rows']:,}"),
    ("Prediction artifact SHA-256", full["sha256"]),
    ("Prediction input identity", full["input_id"]),
], columns=["Verified evidence", "Value"]))
assert full["rows"] == 3 * full["sessions"]
for labels in preview["labels"]:
    items = labels.split()
    assert len(items) == len(set(items)) == 20
    assert all(item.isdecimal() for item in items)
display(preview)


## Decision and limits

A prediction file is an engineering deliverable, not a measured hidden-test score. The portfolio's quality claims come from the controlled temporal evaluation in notebook 09. The complete competition output is generated and validated in full mode; the default notebook proves a small exact native-model replay for convenient review. There is no claim of Kaggle acceptance or a leaderboard position.

The model is frozen during both modes. Rerunning an identical full job checks completed part digests and resumes missing work. A changed test input, retrieval graph, model seal, or feature implementation requires a distinct experiment workspace.


In [ ]:
print(datetime.now(UTC).isoformat(), "inference_notebook_complete",
      f"elapsed_seconds={time.perf_counter()-STARTED:.3f}")
print("Generated output:", destination)
print("Runtime:", {"lightgbm": lgb.__version__, "numpy": np.__version__, "pandas": pd.__version__})
